# Building the RL Research Dataset

**Project:** RL-Based Early Warning of Neurological Deterioration in Sepsis ICU Patients

**Dataset:** MIMIC-IV

## Objective

Construct a research-ready longitudinal ICU dataset by integrating:

- Clinical vital signs
- GCS-based neurological status
- ECG-derived HRV features
- Temporal deterioration labels

The final dataset will be used to formulate the ICU monitoring problem as a Reinforcement Learning task.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Paths
# -----------------------------

PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "mimic-iv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)

Raw data: ../data/raw/mimic-iv
Processed data: ../data/processed


In [2]:
cohort_path = (
    PROCESSED_DIR /
    "sepsis_icu_cohort_demo.csv"
)

sepsis_icu = pd.read_csv(
    cohort_path,
    parse_dates=["intime", "outtime"]
)

print("Sepsis ICU cohort:", sepsis_icu.shape)

display(sepsis_icu.head())

Sepsis ICU cohort: (33, 8)


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10020740,25826145,32145159,Trauma SICU (TSICU),Trauma SICU (TSICU),2150-06-03 20:12:32,2150-06-04 21:05:58,1.037106
1,10039708,23819016,38559363,Trauma SICU (TSICU),Trauma SICU (TSICU),2140-06-18 01:41:00,2140-06-19 21:47:16,1.837685
2,10018081,28861356,38333427,Trauma SICU (TSICU),Trauma SICU (TSICU),2134-08-05 14:53:33,2134-08-07 17:32:43,2.110532
3,10018081,21027282,37293400,Trauma SICU (TSICU),Trauma SICU (TSICU),2133-12-18 17:10:00,2134-01-01 14:44:53,13.899225
4,10003400,23559586,38383343,Coronary Care Unit (CCU),Medical Intensive Care Unit (MICU),2137-08-17 17:36:37,2137-09-02 19:17:11,16.069838


In [3]:
TARGET_ITEMS = {
    "heart_rate": 220045,
    "map": 220052,
    "resp_rate": 220210,
    "gcs_motor": 223901,
    "gcs_verbal": 223900,
    "gcs_eye": 220739,
}

TARGET_ITEMIDS = list(TARGET_ITEMS.values())

print("Target clinical variables:")
for name, itemid in TARGET_ITEMS.items():
    print(f"{name:15s} → {itemid}")

Target clinical variables:
heart_rate      → 220045
map             → 220052
resp_rate       → 220210
gcs_motor       → 223901
gcs_verbal      → 223900
gcs_eye         → 220739


In [4]:
chartevents_path = (
    RAW_DIR /
    "icu" /
    "chartevents.csv.gz"
)

columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "charttime",
    "itemid",
    "value",
    "valuenum",
    "valueuom",
]

print("Loading chartevents...")

chartevents = pd.read_csv(
    chartevents_path,
    compression="gzip",
    usecols=columns,
    low_memory=False
)

print("Total chartevents loaded:", len(chartevents))

Loading chartevents...
Total chartevents loaded: 668862


In [5]:
sepsis_stays = set(
    sepsis_icu["stay_id"].astype(int)
)

clinical_events = chartevents[
    chartevents["stay_id"].isin(sepsis_stays)
    &
    chartevents["itemid"].isin(TARGET_ITEMIDS)
].copy()

clinical_events["charttime"] = pd.to_datetime(
    clinical_events["charttime"]
)

print(
    "Filtered clinical events:",
    clinical_events.shape
)

print(
    "ICU stays:",
    clinical_events["stay_id"].nunique()
)

print(
    "Patients:",
    clinical_events["subject_id"].nunique()
)

Filtered clinical events: (16718, 8)
ICU stays: 33
Patients: 17


In [6]:
item_to_feature = {
    220045: "heart_rate",
    220052: "map",
    220210: "resp_rate",
    223901: "gcs_motor",
    223900: "gcs_verbal",
    220739: "gcs_eye",
}

clinical_events["feature"] = (
    clinical_events["itemid"]
    .map(item_to_feature)
)

clinical_wide = (
    clinical_events
    .pivot_table(
        index=[
            "subject_id",
            "hadm_id",
            "stay_id",
            "charttime"
        ],
        columns="feature",
        values="valuenum",
        aggfunc="last"
    )
    .reset_index()
)

clinical_wide.columns.name = None

print("Clinical wide table:", clinical_wide.shape)

display(clinical_wide.head())

Clinical wide table: (5986, 10)


,subject_id,hadm_id,stay_id,charttime,gcs_eye,gcs_motor,gcs_verbal,heart_rate,map,resp_rate
0,10002428,20321825,34807493,2156-04-30 22:28:00,NaN,NaN,NaN,112.0,NaN,27.0
1,10002428,20321825,34807493,2156-04-30 22:42:00,NaN,NaN,NaN,102.0,NaN,24.0
2,10002428,20321825,34807493,2156-04-30 22:43:00,4.0,5.0,4.0,NaN,NaN,NaN
3,10002428,20321825,34807493,2156-04-30 23:00:00,NaN,NaN,NaN,105.0,NaN,22.0
4,10002428,20321825,34807493,2156-05-01 00:00:00,NaN,NaN,NaN,98.0,NaN,29.0


In [7]:
clinical_wide["gcs_total"] = (
    clinical_wide["gcs_eye"]
    + clinical_wide["gcs_verbal"]
    + clinical_wide["gcs_motor"]
)

clinical_wide["gcs_total"].describe()

count    1098.000000
mean       10.959016
std         3.806768
min         3.000000
25%         9.000000
50%        11.000000
75%        15.000000
max        15.000000
Name: gcs_total, dtype: float64

In [8]:
display(
    clinical_wide[
        [
            "subject_id",
            "stay_id",
            "charttime",
            "heart_rate",
            "map",
            "resp_rate",
            "gcs_total"
        ]
    ].head(20)
)

,subject_id,stay_id,charttime,heart_rate,map,resp_rate,gcs_total
0,10002428,34807493,2156-04-30 22:28:00,112.0,NaN,27.0,NaN
1,10002428,34807493,2156-04-30 22:42:00,102.0,NaN,24.0,NaN
2,10002428,34807493,2156-04-30 22:43:00,NaN,NaN,NaN,13.0
3,10002428,34807493,2156-04-30 23:00:00,105.0,NaN,22.0,NaN
4,10002428,34807493,2156-05-01 00:00:00,98.0,NaN,29.0,NaN
5,10002428,34807493,2156-05-01 01:00:00,87.0,NaN,23.0,NaN
6,10002428,34807493,2156-05-01 02:00:00,86.0,NaN,25.0,NaN
7,10002428,34807493,2156-05-01 03:00:00,89.0,NaN,22.0,NaN
8,10002428,34807493,2156-05-01 04:00:00,86.0,NaN,26.0,NaN
9,10002428,34807493,2156-05-01 05:00:00,82.0,NaN,23.0,NaN


In [9]:
clinical_wide = clinical_wide.merge(
    sepsis_icu[
        [
            "subject_id",
            "hadm_id",
            "stay_id",
            "intime",
            "outtime"
        ]
    ],
    on=[
        "subject_id",
        "hadm_id",
        "stay_id"
    ],
    how="left"
)

clinical_wide["hours_from_icu_admission"] = (
    clinical_wide["charttime"]
    - clinical_wide["intime"]
).dt.total_seconds() / 3600

print(
    clinical_wide[
        "hours_from_icu_admission"
    ].describe()
)

count    5986.000000
mean      120.377308
std       106.356412
min        -0.466667
25%        33.686667
50%        88.320417
75%       187.319444
max       488.225278
Name: hours_from_icu_admission, dtype: float64


In [10]:
clinical_wide["hour"] = (
    clinical_wide[
        "hours_from_icu_admission"
    ]
    .floordiv(1)
    .astype(int)
)

print(
    "Hourly observations:",
    clinical_wide["hour"].nunique()
)

Hourly observations: 488


In [11]:
hourly_clinical = (
    clinical_wide
    .groupby(
        [
            "subject_id",
            "hadm_id",
            "stay_id",
            "hour"
        ],
        as_index=False
    )
    .agg(
        heart_rate=("heart_rate", "mean"),
        map=("map", "mean"),
        resp_rate=("resp_rate", "mean"),
        gcs_eye=("gcs_eye", "last"),
        gcs_verbal=("gcs_verbal", "last"),
        gcs_motor=("gcs_motor", "last"),
        gcs_total=("gcs_total", "last"),
    )
)

print("Hourly clinical dataset:")
print(hourly_clinical.shape)

display(hourly_clinical.head(20))

Hourly clinical dataset:
(4594, 11)


,subject_id,hadm_id,stay_id,hour,heart_rate,map,resp_rate,gcs_eye,gcs_verbal,gcs_motor,gcs_total
0,10002428,20321825,34807493,0,107.0,NaN,25.5,4.0,4.0,5.0,13.0
1,10002428,20321825,34807493,1,105.0,NaN,22.0,NaN,NaN,NaN,NaN
2,10002428,20321825,34807493,2,98.0,NaN,29.0,NaN,NaN,NaN,NaN
3,10002428,20321825,34807493,3,87.0,NaN,23.0,NaN,NaN,NaN,NaN
4,10002428,20321825,34807493,4,86.0,NaN,25.0,NaN,NaN,NaN,NaN
5,10002428,20321825,34807493,5,89.0,NaN,22.0,NaN,NaN,NaN,NaN
6,10002428,20321825,34807493,6,86.0,NaN,26.0,NaN,NaN,NaN,NaN
7,10002428,20321825,34807493,7,82.0,NaN,23.0,NaN,NaN,NaN,NaN
8,10002428,20321825,34807493,8,85.0,NaN,25.0,3.0,3.0,5.0,11.0
9,10002428,20321825,34807493,9,84.0,NaN,24.0,NaN,NaN,NaN,NaN


In [12]:
hourly_clinical = hourly_clinical.sort_values(
    ["stay_id", "hour"]
)

hourly_clinical["previous_gcs"] = (
    hourly_clinical
    .groupby("stay_id")["gcs_total"]
    .shift(1)
)

hourly_clinical["gcs_change"] = (
    hourly_clinical["gcs_total"]
    - hourly_clinical["previous_gcs"]
)

In [14]:
SAE_THRESHOLD = -3
hourly_clinical["sae"] = (
    hourly_clinical["gcs_change"]
    <= SAE_THRESHOLD
).astype(int)

In [17]:
first_event = (
    hourly_clinical[
        hourly_clinical["sae"] == 1
    ]
    .groupby("stay_id")["hour"]
    .min()
    .rename("event_hour")
)

hourly_clinical = hourly_clinical.merge(
    first_event,
    on="stay_id",
    how="left"
)

hourly_clinical["time_to_event"] = (
    hourly_clinical["event_hour"]
    - hourly_clinical["hour"]
)

In [18]:
hourly_clinical["event_occurred"] = (
    hourly_clinical["event_hour"].notna()
).astype(int)

In [19]:
clinical_output = (
    PROCESSED_DIR /
    "clinical_hourly_demo.csv"
)

hourly_clinical.to_csv(
    clinical_output,
    index=False
)

print("Saved:", clinical_output)
print("Shape:", hourly_clinical.shape)

Saved: ../data/processed/clinical_hourly_demo.csv
Shape: (4594, 19)


In [21]:
print("Total hourly rows:", len(hourly_clinical))

print(
    "Rows marked deterioration:",
    hourly_clinical["sae"].sum()
)

print(
    "ICU stays with deterioration:",
    hourly_clinical.loc[
        hourly_clinical["sae"] == 1,
        "stay_id"
    ].nunique()
)

print(
    "Patients with deterioration:",
    hourly_clinical.loc[
        hourly_clinical["sae"] == 1,
        "subject_id"
    ].nunique()
)
display(
    hourly_clinical[
        hourly_clinical["sae"] == 1
    ][
        [
            "subject_id",
            "stay_id",
            "hour",
            "previous_gcs",
            "gcs_total",
            "gcs_change",
            "event_hour",
            "time_to_event"
        ]
    ].head(20)
)

Total hourly rows: 4594
Rows marked deterioration: 1
ICU stays with deterioration: 1
Patients with deterioration: 1


,subject_id,stay_id,hour,previous_gcs,gcs_total,gcs_change,event_hour,time_to_event
4314,10002428,38875437,1,8.0,4.0,-4.0,1.0,0.0


In [22]:
clinical_output = PROCESSED_DIR / "clinical_hourly_demo.csv"

hourly_clinical.to_csv(
    clinical_output,
    index=False
)

print("Saved successfully!")
print("Path:", clinical_output)
print("Shape:", hourly_clinical.shape)

Saved successfully!
Path: ../data/processed/clinical_hourly_demo.csv
Shape: (4594, 19)
